In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib_etl.s3 import etl_input_data_validator
from lib.s3 import etl_input_table_validator
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")

### Transform 

In [0]:
recency_lookback_duration = data_paths.get("recency_lookback_duration", {})
etl_input_table_validator(
    *[silver_transaction_detail, silver_transaction_fiscal_header, silver_master_item], 
    recency_lookback_duration=recency_lookback_duration,
    spark=spark
)


In [0]:
detail = spark.table(silver_transaction_detail)

detail.createOrReplaceTempView("detail")

header_fiscal = spark.table(silver_transaction_fiscal_header)
header_fiscal.createOrReplaceTempView("header_fiscal")

item = spark.table(silver_master_item)
item.createOrReplaceTempView("item")

detail_fiscal = spark.sql("""
    select
            d.*
        ,h.MBRSHP_SID
        ,h.SITE_NBR
        ,h.FISCAL_WEEK_START
        ,h.FISCAL_WEEK_END
        ,i.ARTICLE_DESC
        ,i.MCH4_CD
        ,i.MCH4_DESC
        ,i.MCH3_CD
        ,i.MCH3_DESC
        ,i.MCH2_CD
        ,i.MCH2_DESC
        ,i.MCH1_CD
        ,i.MCH1_DESC
        ,i.MC_DESC
        ,i.AH1_CD
        ,i.AH1_DESC
        ,i.AH2_CD
        ,i.AH2_DESC
        ,i.AH3_CD
        ,i.AH3_DESC
        ,i.AH4_CD
        ,i.AH4_DESC
        ,i.AH5_CD
        ,i.AH5_DESC
        ,i.AH6_CD
        ,i.AH6_DESC
        ,i.BRAND_TYPE
        ,i.EFF_DT
        ,i.EXP_DT
    from detail d
    join header_fiscal h
        on  d.PURCH_DT = h.PURCH_DT and
            d.PURCH_HDR_ID = h.PURCH_HDR_ID
    left join item i
        on  d.GTIN_CD = i.GTIN_CD and
            d.ARTICLE_NBR = i.ARTICLE_NBR
""")
detail_fiscal.createOrReplaceTempView('source')

In [0]:
validations.validate_table(
        spark,
        "intermediate",
        "detail_fiscal",
        config_validation,
        detail_fiscal,
        stats_etl_path
    )

### Merge

In [0]:
detail_fiscal.write.mode("overwrite").saveAsTable(silver_transaction_fiscal_detail)

if archive_flag:
    save_archive(detail_fiscal, silver_transaction_fiscal_detail_archive, run_as_date)